In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

from pathlib import Path

ROOT = Path.cwd().parent
print(ROOT)

DATA_RAW = ROOT/"data/raw"
DATA_PROCESSED = ROOT/"data/processed"

c:\Users\sebas\PycharmProjects\Git\BoxOffice_Oracle


In [2]:
model_df = pd.read_csv(
    DATA_PROCESSED/"the_numbers_model_base_v1.csv"
)

print(model_df.shape)
model_df.head()

(2255, 30)


,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,...,source,production_method,creative_type,production_countries,languages,legs,plot_point,raw_opening_weekend_text,raw_theater_counts_text,raw_domestic_releases_text
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,56061504.0,3602.0,2005-07-08,Wide,...,Based on Comic/Graphic Novel,Live Action,Super Hero,United States,English,2.76,"Friends turned Enemies, Origin Story, Revenge","$56,061,504 (36.2% of total gross)","3,602 opening theaters/3,619 max. theaters, 5....","July 8th, 2005 (Wide) by 20th Century Fox"
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,NaN,19145480.0,3204.0,2005-09-16,Expands Wide,...,Based on Folk Tale/Legend/Fairytale,Stop-Motion Animation,Fantasy,United States,English,2.85,"Arranged Marriage, Friendly Ghost, Romance, Zo...","$19,145,480 (35.1% of total gross)","3,204 opening theaters/3,204 max. theaters, 5....","September 16th, 2005 (Special Engagement) by W..."
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,108435841.0,3661.0,2005-05-19,Wide,...,Original Screenplay,Animation/Live Action,Science Fiction,United States,English,3.82,"Betrayal, Cloning, Cyborg, Death of a Spouse o...","$108,435,841 (26.2% of total gross)","3,661 opening theaters/3,663 max. theaters, 8....","May 19th, 2005 (Wide) by 20th Century Fox Apri..."
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,NaN,23172440.0,3028.0,2004-04-02,Wide,...,Based on Comic/Graphic Novel,Live Action,Super Hero,United States,English,2.57,Demons Source material: Dark Horse Comics,"$23,172,440 (38.9% of total gross)","3,028 opening theaters/3,043 max. theaters, 4....","April 2nd, 2004 (Wide) by Sony Pictures"
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,5935256.0,1603.0,2008-03-07,Wide,...,Based on Real Life Events,Live Action,Dramatization,United Kingdom,English,5.06,"Bank Robbery, Blackmail, Corrupt Cops, Heist, ...","$5,935,256 (19.7% of total gross)","1,603 opening theaters/1,613 max. theaters, 6....","March 7th, 2008 (Wide) by Lionsgate"


In [3]:
model_df["domestic_release_date"] = pd.to_datetime(
    model_df["domestic_release_date"],
    errors="coerce"
)

model_df = model_df[
    model_df["opening_weekend_gross"].notna()
].copy()

model_df["log_opening_weekend_gross"] = np.log1p(
    model_df["opening_weekend_gross"]
)

model_df["release_month"] = model_df["domestic_release_date"].dt.month
model_df["release_day_of_year"] = model_df["domestic_release_date"].dt.dayofyear

In [4]:
crew = pd.read_parquet(DATA_RAW / "title.crew.parquet")
principals = pd.read_parquet(DATA_RAW / "title.principals.parquet")
names = pd.read_parquet(DATA_RAW / "name.basics.parquet")

In [5]:
valid_tconsts = set(model_df["tconst"])

crew = crew[
    crew["tconst"].isin(valid_tconsts)
].copy()

principals = principals[
    principals["tconst"].isin(valid_tconsts)
].copy()

In [6]:
crew["director_id"] = (
    crew["directors"]
    .str.split(",")
    .str[0]
)

crew["writer_id"] = (
    crew["writers"]
    .str.split(",")
    .str[0]
)

In [7]:
model_df = model_df.merge(
    crew[[
        "tconst",
        "director_id",
        "writer_id"
    ]],
    on="tconst",
    how="left"
)

In [8]:
principals['category'].unique()

<ArrowStringArray>
[              'actor',             'actress',            'director',
              'writer',            'producer',            'composer',
     'cinematographer',              'editor',    'casting_director',
 'production_designer',     'archive_footage',                'self',
       'archive_sound']
Length: 13, dtype: str

In [9]:
top_cast = principals[
    principals["ordering"] <= 6
].copy()

top_cast = top_cast[
    top_cast["category"].isin([
        "actor",
        "actress"
    ])
].copy()

In [10]:
top_cast.head()

,tconst,ordering,nconst,category,job,characters
1736507,tt0120667,1,nm0344435,actor,NaN,"[""Reed Richards""]"
1736508,tt0120667,2,nm0004821,actor,NaN,"[""Ben Grimm""]"
1736509,tt0120667,3,nm0262635,actor,NaN,"[""Johnny Storm""]"
1736510,tt0120667,4,nm0004695,actress,NaN,"[""Sue Storm""]"
1736511,tt0120667,5,nm0573037,actor,NaN,"[""Victor Von Doom""]"


In [11]:
top_cast = top_cast.drop_duplicates(
    subset=["tconst", "nconst"]
)

top_cast = (
    top_cast
    .sort_values(["tconst", "ordering"])
)

top_cast["cast_num"] = (
    top_cast.groupby("tconst")
    .cumcount() + 1
)

cast_pivot = (
    top_cast
    .pivot(
        index="tconst",
        columns="cast_num",
        values="nconst"
    )
)

cast_pivot.columns = [
    f"actor_{c}"
    for c in cast_pivot.columns
]

cast_pivot = cast_pivot.reset_index()

In [12]:
cast_pivot.head()

,tconst,actor_1,actor_2,actor_3,actor_4,actor_5,actor_6
0,tt0120667,nm0344435,nm0004821,nm0262635,nm0004695,nm0573037,nm0512934
1,tt0121164,nm0000136,nm0000307,nm0001833,nm0001808,nm0925768,NaN
2,tt0121766,nm0159789,nm0000204,nm0000191,nm0000168,nm0001519,nm0001751
3,tt0167190,nm0000579,nm0427964,nm0004757,nm0000457,nm1140344,nm0734558
4,tt0200465,nm0005458,nm0004787,nm1304386,nm0990547,nm0269077,nm0202810


In [13]:
model_df = model_df.merge(
    cast_pivot,
    on="tconst",
    how="left"
)

In [14]:
name_lookup = names[[
    "nconst",
    "primaryName"
]].copy()

model_df = model_df.merge(
    name_lookup.rename(columns={
        "nconst": "director_id",
        "primaryName": "director_name"
    }),
    on="director_id",
    how="left"
)

model_df = model_df.merge(
    name_lookup.rename(columns={
        "nconst": "writer_id",
        "primaryName": "writer_name"
    }),
    on="writer_id",
    how="left"
)

for i in range(1, 4):

    model_df = model_df.merge(
        name_lookup.rename(columns={
            "nconst": f"actor_{i}",
            "primaryName": f"actor_{i}_name"
        }),
        on=f"actor_{i}",
        how="left"
    )

In [16]:
model_df.head()

,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,...,actor_2,actor_3,actor_4,actor_5,actor_6,director_name,writer_name,actor_1_name,actor_2_name,actor_3_name
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,56061504.0,3602.0,2005-07-08,Wide,...,nm0004821,nm0262635,nm0004695,nm0573037,nm0512934,Tim Story,Mark Frost,Ioan Gruffudd,Michael Chiklis,Chris Evans
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,NaN,19145480.0,3204.0,2005-09-16,Expands Wide,...,nm0000307,nm0001833,nm0001808,nm0925768,NaN,Tim Burton,John August,Johnny Depp,Helena Bonham Carter,Emily Watson
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,108435841.0,3661.0,2005-05-19,Wide,...,nm0000204,nm0000191,nm0000168,nm0001519,nm0001751,George Lucas,George Lucas,Hayden Christensen,Natalie Portman,Ewan McGregor
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,NaN,23172440.0,3028.0,2004-04-02,Wide,...,nm0427964,nm0004757,nm0000457,nm1140344,nm0734558,Guillermo del Toro,Guillermo del Toro,Ron Perlman,Doug Jones,Selma Blair
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,5935256.0,1603.0,2008-03-07,Wide,...,nm0004787,nm1304386,nm0990547,nm0269077,nm0202810,Roger Donaldson,Dick Clement,Jason Statham,Saffron Burrows,Stephen Campbell Moore


In [15]:
model_df.to_csv(DATA_RAW/"default_model_df.csv")